# 03 – HC18 preparation (Phase 2, M4/M5 data)
Builds filled head masks from the sonographer ellipse outlines, derives boxes, audits duplicates, makes the
leakage-safe 70/15/15 group split (exam id ∪ near-duplicate cluster — a proxy for patient grouping, HC18 has no
patient ids) and exports the YOLO dataset. All logic lives in `fetalvision.data.prepare_hc18`; this notebook only runs the CLI and shows the outputs.

Source: HC18 Grand Challenge training set (van den Heuvel et al. 2018, Zenodo 1327317, CC BY 4.0).

## 0. Setup (Colab)
Clone the repo, install the pinned requirements, point `FETALVISION_DATA` at the data folder (Google Drive or a local copy) and log in to W&B (optional – offline mode is used when no key is present).

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone https://github.com/lamooooo1418/FetalVision.git  # TODO: replace with the team repository URL
    %cd FetalVision
    !pip install -q -r requirements.txt && pip install -q -e .
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["FETALVISION_DATA"] = "/content/drive/MyDrive/FetalVision_data"   # must contain hc18/raw/training_set.zip + CSV
else:
    os.environ.setdefault("FETALVISION_DATA", os.path.abspath("../data"))
    %cd ..  # notebook lives in notebooks/, CLIs run from the repo root
print("FETALVISION_DATA =", os.environ["FETALVISION_DATA"])

In [ ]:
# W&B login (optional). Without a key the CLIs log offline; sync later with `wandb sync wandb/offline-run-*`.
import os
if os.environ.get("WANDB_API_KEY"):
    import wandb; wandb.login()
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_MODE=offline")

In [ ]:
!python -m fetalvision.data.prepare_hc18 --config configs/phase2_hc18_data.yaml

In [ ]:
import json; a = json.load(open("results/phase2/hc18_prep/audit.json"))
print({s: (v["n_images"], v["n_groups"]) for s, v in a["per_split"].items()})
print("exact dup sets:", a["exact_duplicates"]["n_sets"], "| near-dup clusters:", a["near_duplicates"]["n_clusters_gt1"], "| mask failures:", a["mask_failures"]["n"])
print("HC from fitted ellipse vs CSV (mm):", a["hc_from_fitted_ellipse_vs_csv_mm"])
for s, v in a["per_split"].items(): print(s, "HC mean/std", round(v["hc_mm"]["mean"],1), round(v["hc_mm"]["std"],1), "px size", round(v["pixel_size_mm"]["mean"],4))

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/hc18_prep/figures/verification_sheet_*.png', 'results/phase2/hc18_prep/figures/mask_failures.png']], [])):
    print(f); display(Image(filename=f, width=900))

### Team action
Open `results/phase2/hc18_prep/annotation_verification.csv` and set `visual_status` to `ok` / `issue` for the 40 sampled images (replaces the earlier `HC18_Visual_Review.csv` with 30 "Needs review" rows).